In [ ]:
# Parameters (will be overridden by URL query params when running with Voila)
endToEndId = "default_entity_id"  # Can be end_to_end_id or entity_id
# Backend base URL comes from environment when running under Voila
# Set CASE_MGMT_BACKEND_URL in the environment; this default is for local dev only.
import os
backendUrl = os.getenv("CASE_MGMT_BACKEND_URL", "http://localhost:3000")
# Shared secret for proxy requests (set JUPYTER_SHARED_SECRET in environment)
JUPYTER_SHARED_SECRET = os.getenv('JUPYTER_SHARED_SECRET')
if JUPYTER_SHARED_SECRET:
    headers = {'x-jupyter-secret': JUPYTER_SHARED_SECRET}
else:
    headers = {}
filter = "Last Month"
# Fallback end-to-end ID for demo/development
fallback_end_to_end_id = "ee4f3638-c42d-4a7e-abec-4c3aff068570"
# Fallback account ID for Benford's Law analysis
benfordAccountId = "cdtrAcct_3a1f3d24fb2046f2a28dc1fa506d6d69"


In [ ]:
try:
    import requests
    import pandas as pd
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
    from IPython.display import display, HTML
    import os
    
    # Set default renderer to basic html/notebook
    pio.renderers.default = "notebook"

    # Output styling
    display(HTML("""
    <style>
        body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
        .metric-card { background: white; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); text-align: center; }
        .metric-value { font-size: 24px; font-weight: bold; color: #111827; }
        .metric-label { font-size: 12px; color: #6B7280; text-transform: uppercase; letter-spacing: 0.05em; }
        .grid-container { display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px; }
        .table-container { margin-top: 20px; overflow-x: auto; }
        table { width: 100%; border-collapse: collapse; font-size: 14px; }
        th { text-align: left; padding: 12px; border-bottom: 1px solid #e5e7eb; color: #6B7280; font-weight: 600; }
        td { padding: 12px; border-bottom: 1px solid #e5e7eb; color: #111827; }
        tr:last-child td { border-bottom: none; }
    </style>
    """))
except Exception as e:
    print(f"Import Error: {e}")

In [ ]:
# Fetch Data Helper - supports both entity_id and end_to_end_id via backend smart detection
def fetch_transaction_history(id_value, backend_url, time_filter=None):
    """
    Fetch transaction history using the backend endpoint.
    Backend auto-detects if id_value is:
    - UUID format (end_to_end_id): returns single transaction with all 4 entity perspectives
    - Entity ID format (account/counterparty): returns transaction history for that entity
    """
    url = f"{backend_url}/api/v1/jupyter/proxy/transaction-history/{id_value}"
    params = {'tenantId': 'DEFAULT'}
    if time_filter:
        params['filter'] = time_filter
    
    try:
        response = requests.get(url, params=params, headers=headers)
        if not response.ok:
            try:
                error_data = response.json()
                error_msg = error_data.get('message', response.text)
                print(f"Request to {url} failed: {response.status_code} {response.reason}")
                print(f'Error message: {error_msg}')
            except Exception:
                print(f"Request to {url} failed: {response.status_code} {response.reason}")
                print('Response body:', response.text)
            return None
        return response.json()
    except Exception as e:
        import traceback
        print(f"Exception during fetch_transaction_history: {e}")
        traceback.print_exc()
        return None

# Main Fetch Logic with Fallback
# Ensure variables exist even when fetch fails
data = None
summary = {}
timeline = []
volume_dist = []
cumulative = []
recent = []
df_timeline = pd.DataFrame()
df_volume = pd.DataFrame()
df_cumulative = pd.DataFrame()
df_recent = pd.DataFrame()

try:
    # 1. Try requested endToEndId (backend auto-detects if it's transaction or entity ID)
    if endToEndId and endToEndId != "default_entity_id":
        data = fetch_transaction_history(endToEndId, backendUrl, filter)

    # 2. If Failed/Empty or Default, try fallback end_to_end_id (Dev environment only)
    if not data:
        print(f"Trying fallback end_to_end_id: {fallback_end_to_end_id}")
        data = fetch_transaction_history(fallback_end_to_end_id, backendUrl)
        
    if data:
        summary = data.get('summary', {})
        timeline = data.get('timeline', [])
        volume_dist = data.get('volumeDistribution', [])
        # Accept alternative keys
        if not volume_dist:
            volume_dist = data.get('volume_dist') or data.get('volume') or []
        cumulative = data.get('cumulative', [])
        recent = data.get('recentTransactions', [])
        
        df_timeline = pd.DataFrame(timeline)
        if not df_timeline.empty:
            df_timeline['date'] = pd.to_datetime(df_timeline['date'])

        # Normalise volume distribution from backend (bucketStart/transactionCount)
        df_volume = pd.DataFrame(volume_dist)

        if not df_volume.empty:
            if 'bucketStart' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['bucketStart'])
            elif 'date' in df_volume.columns:
                df_volume['date'] = pd.to_datetime(df_volume['date'])
            else:
                df_volume['date'] = pd.NaT

            # Ensure we have a transaction-count column for charts
            if 'transactionCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['transactionCount']
            elif 'bucket_tx_count' in df_volume.columns:
                df_volume['txCount'] = df_volume['bucket_tx_count']
            elif 'txCount' in df_volume.columns:
                df_volume['txCount'] = df_volume['txCount']
            elif 'count' in df_volume.columns:
                df_volume['txCount'] = df_volume['count']
            else:
                df_volume['txCount'] = 0
            
        df_cumulative = pd.DataFrame(cumulative)
        if not df_cumulative.empty:
            df_cumulative['date'] = pd.to_datetime(df_cumulative['date'])
            
        df_recent = pd.DataFrame(recent)
    else:
        display(HTML(f"""
        <div style='color: #ef4444; padding: 20px; border: 2px solid #fee2e2; border-radius: 8px; background: #fef2f2;'>
            <h3 style='margin-top: 0;'>No Data Available</h3>
            <p>Could not fetch transaction data from the backend.</p>
            <p><strong>Backend URL:</strong> {backendUrl}</p>
            <p><strong>Query ID:</strong> {endToEndId if endToEndId != 'default_entity_id' else fallback_end_to_end_id}</p>
            <p>Please check:</p>
            <ul>
                <li>Backend server is running</li>
                <li>The ID exists in the database</li>
                <li>Database connection is working</li>
                <li>Check backend logs for detailed error messages</li>
            </ul>
        </div>
        """))
            
except Exception as e:
    import traceback
    display(HTML(f"<div style='color: red; padding: 20px; border: 1px solid red; border-radius: 5px;'>Error fetching data: {str(e)}</div>"))
    print(f"Backend URL: {backendUrl}")
    traceback.print_exc()


In [ ]:
if data:
    # Display Metrics
    total_vol = f"${summary.get('totalVolume', 0):,.2f}"
    total_tx = f"{summary.get('totalTransactions', 0):,}"
    alerts = f"{summary.get('alertsTriggered', 0)}"
    investigated = f"{summary.get('investigated', 0)}"

    display(HTML(f"""
    <div class=\"grid-container\">
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Volume</div>
            <div class=\"metric-value\">{total_vol}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Total Transactions</div>
            <div class=\"metric-value\">{total_tx}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Alerts Triggered</div>
            <div class=\"metric-value\">{alerts}</div>
        </div>
        <div class=\"metric-card\">
            <div class=\"metric-label\">Investigated</div>
            <div class=\"metric-value\">{investigated}</div>
        </div>
    </div>
    """))

In [ ]:
if data and not df_timeline.empty:
    from datetime import datetime, timedelta
    import calendar
    
    # Sort timeline by date chronologically
    df_timeline = df_timeline.sort_values('date')
    
    # Determine the month to display (from the first transaction)
    min_date = df_timeline['date'].min()
    
    # Get the first and last day of the month
    first_day = min_date.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    last_day_num = calendar.monthrange(min_date.year, min_date.month)[1]
    last_day = min_date.replace(day=last_day_num, hour=23, minute=59, second=59, microsecond=999999)
    
    # Generate complete date range for the entire month (daily granularity)
    date_range = pd.date_range(start=first_day, end=last_day, freq='D')
    df_complete_month = pd.DataFrame({'date': date_range})
    
    # Prepare transaction data with date-only for aggregation
    df_timeline_daily = df_timeline.copy()
    df_timeline_daily['date_only'] = df_timeline_daily['date'].dt.normalize()
    
    # Aggregate transactions by day: sum amounts, count transactions, track alerts
    df_daily_agg = (
        df_timeline_daily.groupby('date_only')
        .agg(
            amount=('amount', 'sum'),
            txCount=('transactionId', 'count'),
            hasAlert=('isAlerted', 'max')  # True if any transaction that day was alerted
        )
        .reset_index()
        .rename(columns={'date_only': 'date'})
    )
    
    # Merge complete month with actual transaction data
    df_normalized = df_complete_month.merge(df_daily_agg, on='date', how='left')
    
    # Fill missing values with 0
    df_normalized['amount'] = df_normalized['amount'].fillna(0)
    df_normalized['txCount'] = df_normalized['txCount'].fillna(0).astype(int)
    df_normalized['hasAlert'] = df_normalized['hasAlert'].fillna(False).astype(bool)
    
    # Calculate X-axis range with padding (convert to strings for Plotly)
    x_range_start = (first_day - timedelta(days=0.5)).strftime('%Y-%m-%d')
    x_range_end = (last_day + timedelta(days=0.5)).strftime('%Y-%m-%d')
    
    # Separate days with and without alerts for visualization
    df_alert_days = df_normalized[df_normalized['hasAlert'] == True]
    df_normal_days = df_normalized[(df_normalized['hasAlert'] == False) & (df_normalized['amount'] > 0)]
    
    # Create Charts: 3 rows -> Timeline (dual axis), Cumulative, Volume Distribution
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=False,  # Changed to False to control each chart independently
        vertical_spacing=0.20,  # Increased spacing between charts
        subplot_titles=(
            "Transaction Timeline",
            "Cumulative Value",
            "Volume Distribution"
        ),
        row_heights=[0.4, 0.3, 0.3],
        specs=[
            [{"secondary_y": True}],  # Row 1: dual y-axis
            [{}],                      # Row 2
            [{}],                      # Row 3
        ],
    )

    # 1. Transaction Timeline - Amount Line (Blue, smooth curve)
    fig.add_trace(
        go.Scatter(
            x=df_normalized['date'],
            y=df_normalized['amount'],
            mode='lines',
            name='Amount ($)',
            line=dict(color='#3B82F6', width=2.5, shape='spline'),
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Amount: $%{y:,.2f}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
        secondary_y=False,
    )
    
    # Add markers for days with normal transactions (blue dots)
    if not df_normal_days.empty:
        fig.add_trace(
            go.Scatter(
                x=df_normal_days['date'],
                y=df_normal_days['amount'],
                mode='markers',
                name='Transaction',
                marker=dict(size=8, color='#3B82F6', line=dict(width=1, color='white')),
                hovertemplate=(
                    "<b>%{x|%b %d, %Y}</b><br>" +
                    "Amount: $%{y:,.2f}<br>" +
                    "Status: Normal<br>" +
                    "<extra></extra>"
                ),
                showlegend=True,
            ),
            row=1,
            col=1,
            secondary_y=False,
        )
    
    # Add markers for alert days (red dots)
    if not df_alert_days.empty:
        fig.add_trace(
            go.Scatter(
                x=df_alert_days['date'],
                y=df_alert_days['amount'],
                mode='markers',
                name='Alert Triggered',
                marker=dict(
                    size=10, 
                    color='#EF4444',
                    line=dict(width=2, color='white'),
                    symbol='circle'
                ),
                hovertemplate=(
                    "<b>%{x|%b %d, %Y}</b><br>" +
                    "Amount: $%{y:,.2f}<br>" +
                    "Status: <span style='color:#EF4444;font-weight:bold;'>⚠ ALERT</span><br>" +
                    "<extra></extra>"
                ),
                showlegend=True,
            ),
            row=1,
            col=1,
            secondary_y=False,
        )

    # 1b. Transaction Count Line (Green, smooth curve on right Y-axis)
    fig.add_trace(
        go.Scatter(
            x=df_normalized['date'],
            y=df_normalized['txCount'],
            mode='lines+markers',
            name='Transaction Count',
            line=dict(color='#10B981', width=2.5, shape='spline'),
            marker=dict(size=6, color='#10B981', line=dict(width=1, color='white')),
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Count: %{y}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=1,
        col=1,
        secondary_y=True,
    )

    # 2. Cumulative Value (Running Total)
    df_normalized['cumulativeAmount'] = df_normalized['amount'].cumsum()
    
    fig.add_trace(
        go.Scatter(
            x=df_normalized['date'],
            y=df_normalized['cumulativeAmount'],
            fill='tozeroy',
            mode='lines',
            name='Cumulative Value',
            line=dict(color='#10B981', width=2, shape='spline'),
            fillcolor='rgba(16, 185, 129, 0.1)',
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Cumulative: $%{y:,.2f}<br>" +
                "<extra></extra>"
            ),
            connectgaps=False,
        ),
        row=2,
        col=1,
    )

    # 3. Transaction Volume Distribution (Bar chart of counts)
    fig.add_trace(
        go.Bar(
            x=df_normalized['date'],
            y=df_normalized['txCount'],
            name='Volume Distribution',
            marker_color='#6366F1',
            hovertemplate=(
                "<b>%{x|%b %d, %Y}</b><br>" +
                "Transactions: %{y}<br>" +
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    # Layout & axes styling - Modern fintech dashboard style
    fig.update_layout(
        height=900,  # Increased height to accommodate spacing
        showlegend=False,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="#E5E7EB",
            borderwidth=1,
            font=dict(size=11)
        ),
        margin=dict(l=60, r=60, t=60, b=40),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        hovermode='x unified',
        font=dict(
            family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif",
            color="#111827"
        ),
    )

    # Grid lines with subtle styling
    fig.update_xaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(229, 231, 235, 0.5)',
        showline=True,
        linewidth=1,
        linecolor='#E5E7EB',
        mirror=False
    )
    fig.update_yaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(229, 231, 235, 0.5)',
        showline=True,
        linewidth=1,
        linecolor='#E5E7EB',
        mirror=False,
        zeroline=True,
        zerolinewidth=1,
        zerolinecolor='#E5E7EB'
    )

    # Row 1: Transaction Timeline - X-axis configuration
    fig.update_xaxes(
        title_text="<b>Date (Full Month)</b>", 
        row=1, 
        col=1,
        tickformat='%b %d',
        tickangle=-45,
        tickmode='auto',
        nticks=min(last_day_num, 31),
        range=[x_range_start, x_range_end],
        tickfont=dict(size=9),
        title_font=dict(size=12, color="#6B7280"),
        fixedrange=False
    )
    fig.update_yaxes(
        title_text="<b>Amount ($)</b>", 
        row=1, 
        col=1, 
        secondary_y=False,
        title_font=dict(size=12, color="#3B82F6")
    )
    fig.update_yaxes(
        title_text="<b>Transaction Count</b>", 
        row=1, 
        col=1, 
        secondary_y=True,
        title_font=dict(size=12, color="#10B981")
    )
    
    # Row 2: Cumulative chart - X-axis configuration
    fig.update_xaxes(
        title_text="<b>Date (Full Month)</b>", 
        row=2, 
        col=1,
        tickformat='%b %d',
        tickangle=-45,
        tickmode='auto',
        nticks=min(last_day_num, 31),
        range=[x_range_start, x_range_end],
        tickfont=dict(size=9),
        title_font=dict(size=12, color="#6B7280"),
        fixedrange=False
    )
    fig.update_yaxes(
        title_text="<b>Cumulative Amount ($)</b>", 
        row=2, 
        col=1,
        title_font=dict(size=12, color="#10B981")
    )
    
    # Row 3: Volume distribution - X-axis configuration
    fig.update_xaxes(
        title_text="<b>Date (Full Month)</b>", 
        row=3, 
        col=1,
        tickformat='%b %d',
        tickangle=-45,
        tickmode='auto',
        nticks=min(last_day_num,31),
        range=[x_range_start, x_range_end],
        tickfont=dict(size=9),
        title_font=dict(size=12, color="#6B7280"),
        fixedrange=False
    )
    fig.update_yaxes(
        title_text="<b>Count</b>", 
        row=3, 
        col=1,
        title_font=dict(size=12, color="#6366F1")
    )

    # Add crosshair effect with spike lines
    fig.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikecolor='#9CA3AF', spikethickness=1)
    fig.update_yaxes(showspikes=True, spikemode='across', spikesnap='cursor', spikecolor='#9CA3AF', spikethickness=1)

    fig.show()


In [ ]:
# Fetch Benford's Law Data and Create Analysis Chart
try:
    from datetime import datetime, timedelta
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)

    # Use benford fallback for account-based analysis (Benford requires account ID)
    target_account_id = benfordAccountId

    benford_url = f"{backendUrl}/api/v1/jupyter/proxy/lake/analytics/benford/account/{target_account_id}"
    benford_params = {
        'tenantId': 'DEFAULT',
        'from': start_date.strftime('%Y-%m-%d'),
        'to': end_date.strftime('%Y-%m-%d')
    }

    try:
        benford_response = requests.get(benford_url, params=benford_params, headers=headers, timeout=20)
        
        if not benford_response.ok:
            print(f"Benford request failed: {benford_response.status_code} {benford_response.reason}")
            try:
                print('Response body:', benford_response.text)
            except Exception:
                pass
            benford_data = None
        else:
            benford_response.raise_for_status()
            benford_data = benford_response.json()

    except Exception as e:
        import traceback
        print("Exception during Benford fetch:", e)
        traceback.print_exc()
        benford_data = None

    # Process benford_data if present
    expected_benford = []
    actual_benford = []
    digits = [1,2,3,4,5,6,7,8,9]

    if benford_data and isinstance(benford_data, dict) and 'expected' in benford_data and 'actual' in benford_data:
        exp_dict = benford_data['expected']
        act_dict = benford_data['actual']
        sample_size = float(benford_data.get('sampleSize', 0) or 0)
        for d in digits:
            d_str = str(d)
            expected_benford.append(float(exp_dict.get(d_str, 0)) * 100)
            count = float(act_dict.get(d_str, 0))
            actual_benford.append((count / sample_size * 100) if sample_size > 0 else 0)
    else:
        if benford_data is None:
            print('No Benford data returned from backend')
        else:
            print('Benford response format unexpected:', type(benford_data))

    # Ensure arrays of length 9 for plotting
    if len(expected_benford) != 9:
        expected_benford = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
    if len(actual_benford) != 9:
        actual_benford = [0] * 9

except Exception as e:
    import traceback
    print(f"Benford analysis error: {e}")
    traceback.print_exc()


In [ ]:
# Benford chart and table (always render; fallbacks applied earlier)
try:
    import plotly.graph_objects as go
    import pandas as pd

    digits = [1,2,3,4,5,6,7,8,9]
    # Ensure expected_benford and actual_benford exist
    try:
        exp = expected_benford
        act = actual_benford
    except NameError:
        exp = [30.1, 17.6, 12.5, 9.7, 7.9, 6.7, 5.8, 5.1, 4.6]
        act = [0]*9

    fig = go.Figure()
    fig.add_trace(go.Bar(x=digits, y=exp, name='Expected Benford', marker_color='#10B981'))
    fig.add_trace(go.Bar(x=digits, y=act, name='Actual', marker_color='#6366F1'))
    fig.update_layout(
        barmode='group', 
        title='Benfords Law Distribution Analysis', 
        xaxis_title='First Digit', 
        yaxis_title='Frequency (%)', 
        height=420,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.4,
            xanchor="center",
            x=0.5
        )
    )
    fig.show()

    # Show a small summary table
    try:
        df_b = pd.DataFrame({'digit': digits, 'expected_pct': exp, 'actual_pct': act})
        display(df_b.style.hide_index())
    except Exception:
        pass
except Exception as e:
    import traceback
    print('Error rendering Benford chart/table:', e)
    traceback.print_exc()



In [ ]:
if data and not df_recent.empty:
    display(HTML("<h3>Recent Transactions</h3>"))

    # Normalise status for display, but keep the real values
    if 'status' in df_recent.columns:
        def _format_status(v):
            # Backend sends this as a list of labels like ["Alert", "Investigated"]
            if isinstance(v, (list, tuple)):
                return ", ".join(str(x) for x in v)
            if v is None or (isinstance(v, float) and pd.isna(v)):
                return ""
            return str(v)

        df_recent['status'] = df_recent['status'].apply(_format_status)

    # Select and rename columns for display
    cols = ['date', 'type', 'counterparty', 'amount', 'currency', 'status']
    # Ensure cols exist
    valid_cols = [c for c in cols if c in df_recent.columns]
    display_df = df_recent[valid_cols].copy()
    
    # Format Amount
    if 'amount' in display_df.columns:
        display_df['amount'] = display_df['amount'].apply(lambda x: f"{x:,.2f}")
        
    # Render HTML table with styling
    html = display_df.to_html(index=False, classes='table')
    display(HTML(f"<div class='table-container'>{html}</div>"))